In [20]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import Dataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [21]:
def add_gaussian_noise(df:pd.DataFrame, columns:List[str], mean:float = 0.0, std:float = 0.1):
    """
    Adds Gaussian noise to specified columns in a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to which noise will be added.
        columns (list): A list of column names in the DataFrame to which noise will be added.
        mean (int, optional): The mean of the Gaussian distribution. Defaults to 0.
        std (float, optional): The standard deviation of the Gaussian distribution. Defaults to 0.1.

    Returns:
        pandas.DataFrame: The DataFrame with added Gaussian noise to specified columns.
    """
    for col in columns:
        df[col] = df[col] + np.random.normal(loc=mean, scale=std, size=len(df))
    return df

In [22]:
data = np.arange(0, 100, 0.01)
data_df = pd.DataFrame(data, columns=['x'])
data_df['y'] = data_df['x']

scaler = MinMaxScaler()
data_df = pd.DataFrame(scaler.fit_transform(data_df), columns=['x', 'y'])

In [23]:
data_df

,x,y
0,0.0000,0.0000
1,0.0001,0.0001
2,0.0002,0.0002
3,0.0003,0.0003
4,0.0004,0.0004
...,...,...
9995,0.9996,0.9996
9996,0.9997,0.9997
9997,0.9998,0.9998
9998,0.9999,0.9999


In [24]:
data_original = data_df.copy()
for sigma in np.arange(0.01, 0.012, 0.01):
        print(1)
        df_noisy = add_gaussian_noise(
                df=data_original.copy(),
                columns=['x'],
                mean=0.0,
                std=sigma
        )
        data_df = pd.concat([data_df, df_noisy])

1


In [25]:
data_df = df_noisy

In [26]:
data_df

,x,y
0,0.015321,0.0000
1,-0.002859,0.0001
2,-0.000089,0.0002
3,0.004884,0.0003
4,-0.002896,0.0004
...,...,...
9995,1.003554,0.9996
9996,1.012572,0.9997
9997,1.001934,0.9998
9998,0.994379,0.9999


In [27]:
batch_size = 168 # 1 week of data (24*7)

In [28]:
# data_df = data_df.sample(fr
# 
# ac=1).reset_index(drop=True)
print(data_df)
limit = int(len(data_df)%batch_size)
data_df = data_df.iloc[:-limit]

             x       y
0     0.015321  0.0000
1    -0.002859  0.0001
2    -0.000089  0.0002
3     0.004884  0.0003
4    -0.002896  0.0004
...        ...     ...
9995  1.003554  0.9996
9996  1.012572  0.9997
9997  1.001934  0.9998
9998  0.994379  0.9999
9999  0.996312  1.0000

[10000 rows x 2 columns]


In [29]:
data_df

,x,y
0,0.015321,0.000000
1,-0.002859,0.000100
2,-0.000089,0.000200
3,0.004884,0.000300
4,-0.002896,0.000400
...,...,...
9907,0.982241,0.990799
9908,0.974634,0.990899
9909,0.984846,0.990999
9910,1.014876,0.991099


In [30]:
# Create Datasets for DataLoaders
noisy_dataset = Dataset(
    input_df=data_df['x'],
    target_df=data_df['y']
)

# Create the dataloaders
train_dataloader = DataLoader(noisy_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [31]:
denoiser = DenoisingAutoencoder(
    input_dim=batch_size,
    latent_dim=64,
).to(device)

In [32]:
for x in train_dataloader:
    print(x)
    break

[tensor([ 0.7335,  0.8770,  0.2825,  0.5421,  0.6989,  0.4987,  0.7023,  0.5514,
         0.5925,  0.1876,  0.6082,  0.4983,  0.1794,  0.7261,  0.9630,  0.1003,
         0.2761,  0.6577,  0.2451,  0.2600,  0.4521,  0.7667,  0.3922,  0.4809,
         0.6126,  0.1941,  0.1703,  0.4118,  0.3753,  0.8141,  0.6583,  0.4450,
         0.9114,  0.6698,  0.7951,  0.7840,  0.3530,  0.7784,  0.3655,  0.6709,
         0.4119,  0.0568,  0.2060,  0.4545,  0.4293,  0.6855,  0.4501,  0.1001,
         0.6584,  0.0181,  0.2164,  0.4143,  0.2605,  0.7391,  0.4840,  0.0227,
         0.7006,  0.6026,  0.6647,  0.7730,  0.2821,  0.0382,  0.8182,  0.3719,
         0.6612,  0.3026,  0.6191,  0.8510,  0.0571,  0.6881,  0.9370,  0.8838,
         0.1429,  0.5393,  0.8665,  0.1401,  0.5599,  0.5556,  0.7230,  0.3586,
         0.1861,  0.3657,  0.1473,  0.6734,  0.8062,  0.3523,  0.3149,  0.3646,
         0.1882,  0.5066,  0.8861,  0.8122,  0.0621,  0.6077,  0.9745,  0.9202,
         0.8684,  0.1805,  0.1225,  0.2

In [33]:
data_df

,x,y
0,0.015321,0.000000
1,-0.002859,0.000100
2,-0.000089,0.000200
3,0.004884,0.000300
4,-0.002896,0.000400
...,...,...
9907,0.982241,0.990799
9908,0.974634,0.990899
9909,0.984846,0.990999
9910,1.014876,0.991099


In [34]:
# Set model parameters and create the model Trainer object
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(denoiser.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=denoiser,
    train_generator=train_dataloader,
    val_generator=train_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(
        CHECKPOINT_PATH, 'denoising_autoencoder'
    ),
)

In [35]:
# denoiser = torch.load(os.path.join(CHECKPOINT_PATH, 'denoising_autoencoder.pth'))

In [36]:
# Train the model
model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)


Training is started!


Epochs loop:  20%|▏| 100/500 [02:08<08:32,  1.28s/epoch, Train loss: 0.0819854689604145 - Va

Training stopped as validation loss did not improve for 15 epochs.


In [37]:
y_pred_noisy = trainer_basic.eval_dataloader(train_dataloader)
flattened_tensor = torch.cat(y_pred_noisy).view(-1)
len(flattened_tensor)

9912

In [38]:
# Show the metrics
gt_values = data_df['y'].values
predicted_values = flattened_tensor.cpu().detach().numpy()

mae = mean_absolute_error(gt_values, predicted_values)
mse = mean_squared_error(gt_values, predicted_values)
rmse = np.sqrt(mse)
# mape = np.mean(np.abs(gt_values - predicted_values) / gt_values) * 100
r_squared = r2_score(gt_values, predicted_values)
nn_benchmark = {
    'nn': {
        "MSE: ": mse,
        "RMSE: ": rmse,
        "MAE: ": mae,
        # "MAPE: ": mape,
        "R2: ": r_squared
    }
}
nn_benchmark

{'nn': {'MSE: ': 0.08300265680698475,
  'RMSE: ': 0.288101816736696,
  'MAE: ': 0.24851536204334007,
  'R2: ': -0.013593447401432313}}